## Text input

https://platform.openai.com/docs/models

In [7]:
from dotenv import load_dotenv



load_dotenv()

True

In [8]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("meta-llama/llama-4-scout-17b-16e-instruct", model_provider="groq")

agent = create_agent(
    model=model,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [9]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The Moon, a natural satellite that has been terraformed and colonized by humanity. Its capital city is called Lunaria.

**Location:** Lunaria is situated on the Moon's surface, specifically on the southeastern edge of the Mare Imbrium, one of the largest dark basaltic plains on the Moon. The city's coordinates are 32.5° N latitude and 15.5° W longitude.

**Geography and Climate:** Lunaria is nestled within a vast, circular impact crater, approximately 500 kilometers in diameter. The crater's rim provides a natural barrier against the harsh lunar environment, protecting the city from extreme temperatures, radiation, and meteorite impacts. The city's terrain is relatively flat, with gentle slopes and a few scattered hills. The lunar regolith has been terraformed to create a breathable atmosphere, with a stable temperature range between -20°C to 20°C (-4°F to 68°F).

**Cityscape:** Lunaria is a marvel of modern engineering, with sleek, curved skyscrapers and habitats that blend seamlessly

## Image input

In [10]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [13]:
print(uploader.value)

({'name': 'moon.png', 'type': 'image/png', 'size': 358916, 'content': <memory at 0x1107b8dc0>, 'last_modified': datetime.datetime(2026, 3, 30, 5, 20, 2, 330000, tzinfo=datetime.timezone.utc)},)


In [14]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [15]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

The capital city of Xeridia, named Aethereia, is a futuristic metropolis built on a vast, arid planet. The city is nestled within a valley surrounded by rugged mountains, with the imposing structure of the Celestial Spires dominating the skyline. These towering spires are not only architectural marvels but also serve as advanced energy harvesters, capturing and converting the planet's unique energy signature into power for the city.

**Geography and Climate:**
- **Location:** Aethereia is situated in the heart of the Xeridian Valley, a large, naturally formed depression in the planet's surface. This valley provides a natural barrier against the harsh, desert-like conditions that cover much of the planet.
- **Climate:** The climate within the valley is surprisingly temperate, thanks to advanced atmospheric processing and the energy shield that encases the city. Outside, the planet's climate is harsh, with extreme temperatures and frequent sandstorms.

**Infrastructure:**
- **The Celesti

## Audio input

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

In [ ]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

## Video input (frame extraction + Groq vision model)

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

video_uploader = FileUpload(accept='.mp4,.mov,.avi', multiple=False)
display(video_uploader)

In [ ]:
import cv2
import base64
import tempfile
import os

# Save uploaded video to a temp file
uploaded_video = video_uploader.value[0]
video_bytes = bytes(uploaded_video["content"])

tmp_path = os.path.join(tempfile.gettempdir(), "uploaded_video.mp4")
with open(tmp_path, "wb") as f:
    f.write(video_bytes)

# Extract frames at 1 frame per second
cap = cv2.VideoCapture(tmp_path)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_interval = int(fps)  # 1 frame per second
max_frames = 5  # limit to keep within token limits

frames_b64 = []
frame_count = 0

while cap.isOpened() and len(frames_b64) < max_frames:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % frame_interval == 0:
        _, buffer = cv2.imencode(".jpg", frame)
        frames_b64.append(base64.b64encode(buffer).decode("utf-8"))
    frame_count += 1

cap.release()
print(f"Extracted {len(frames_b64)} frames from video")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# Use Groq's free vision model (llama-3.2-90b-vision-preview)
vision_model = init_chat_model(model="llama-3.2-90b-vision-preview", model_provider="groq", temperature=0)

agent = create_agent(
    model=vision_model,
    system_prompt="You are a helpful assistant that analyzes video content from extracted frames.",
)

# Build message with all frames
content = [{"type": "text", "text": "These are frames extracted from a video (1 per second). Describe what is happening in the video."}]
for i, frame_b64 in enumerate(frames_b64):
    content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{frame_b64}"}})

video_question = HumanMessage(content=content)

response = agent.invoke({"messages": [video_question]})
print(response['messages'][-1].content)